# Hyperparameter transfer

## Basic imports

In [ ]:
import math
import time
import numpy as np

from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms

## Question 1: Spectral norm of Gaussian matrix

In [ ]:
d = 100

M = torch.randn(size=(d, d), device="cuda")
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

## Question 2: Spectral norm of orthogonal matrix

In [ ]:
from torch.nn.init import orthogonal_

d = 100

M = torch.zeros(size=(d, d), device="cuda")
orthogonal_(M)  # this line resamples M to be a random semi-orthogonal matrix
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

## Question 3: Power iteration

In [ ]:
def spectral_norm(A, n_steps=10):
    v = torch.randn(A.shape[1], device=A.device)
    for _ in range(n_steps):
        v /= v.norm()
        v = A @ v @ A
    return v.norm().sqrt()


d = 2000
M = torch.randn(size=(d, d), device="cuda")

torch.cuda.synchronize()
t0 = time.time()
spec_norm = spectral_norm(M).item()
t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

torch.cuda.synchronize()
t0 = time.time()
spec_norm = torch.linalg.matrix_norm(M, ord=2).item()
t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

## Question 4: Learning rate transfer across width and depth
You only need to modify the two blocks of code that are marked, and then choose the learning rate `eta` for the large width training run.

In [ ]:
batch_size = 128

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

trainset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
testset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, pin_memory=True
)

## Define the MLP architecture

In [ ]:
class MLP(nn.Module):
    def __init__(self, depth, width):
        super(MLP, self).__init__()

        self.initial = nn.Linear(3072, width, bias=False)
        self.layers = nn.ModuleList(
            [nn.Linear(width, width, bias=False) for _ in range(depth - 2)]
        )
        self.final = nn.Linear(width, 10, bias=False)

        self.nonlinearity = lambda x: F.relu(x) * math.sqrt(2)

    def forward(self, x):
        x = x.view(x.shape[0], -1)

        x = self.initial(x)
        x = self.nonlinearity(x)

        for layer in self.layers:
            x = layer(x)
            x = self.nonlinearity(x)

        return self.final(x)

## Define the train and test loop

In [ ]:
def loop(net, train, eta):
    dataloader = train_loader if train else test_loader
    description = "Training... " if train else "Testing... "

    acc_list = []

    for data, target in tqdm(dataloader, desc=description):
        data, target = data.cuda(), target.cuda()
        output = net(data)

        loss = (
            output.logsumexp(dim=1).mean()
            - output[range(target.shape[0]), target].mean()
        )  # cross-entropy loss
        acc = (output.max(dim=1)[1] == target).sum() / target.shape[0]  # accuracy
        acc_list.append(acc.item())

        if train:
            loss.backward()

            for p in net.parameters():
                fan_out, fan_in = p.shape
                update = torch.sign(p.grad)
                ## START BLOCK that you should modify
                update /= torch.norm(update)
                ## END BLOCK that you should modify
                p.data -= eta * update
            net.zero_grad()

    return np.mean(acc_list)

## Sweep the learning rate at small width



In [ ]:
depth = 5
width = 32


def initialize_matrix(p):
    fan_out, fan_in = p.shape
    ## START BLOCK that you should modify ##
    p.data = (
        math.sqrt(fan_out)
        * torch.randn_like(torch.nn.init.orthogonal_(p))
        / math.sqrt(fan_in)
    )
    ## END BLOCK that you should modify


for eta in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    print(f"Training at {width=}, {depth=}, {eta=}")

    net = MLP(depth, width).cuda()

    print("\nNetwork tensor shapes are:\n")
    for name, p in net.named_parameters():
        print(p.shape, "\t", name)
        initialize_matrix(p)

    train_acc = loop(net, train=True, eta=eta)
    test_acc = loop(net, train=False, eta=None)

    print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
    print("===================================================================\n")

## Run the best learning rate at large width

In [ ]:
depth = 5
width = 4096

# START BLOCK you should set this to the best value of eta from the previous cell
eta = 0.001
# END BLOCK

print(f"Training at {width=}, {depth=}, {eta=}")

net = MLP(depth, width).cuda()

print("\nNetwork tensor shapes are:\n")
for name, p in net.named_parameters():
    print(p.shape, "\t", name)
    initialize_matrix(p)

train_acc = loop(net, train=True, eta=eta)
test_acc = loop(net, train=False, eta=None)

print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
print("===================================================================\n")